In [1]:
import pandas as pd
import akshare as ak
STOCKS = [
    ("600519", "贵州茅台"),
    ("601318", "中国平安"),
    ("300750", "宁德时代"),
    ("002594", "比亚迪"),
]
START, END = "20230101", "20241231"
VOL_RATIO_MIN = 1.2
def fetch_data(code):
    df = ak.stock_zh_a_hist(symbol=code, period="daily",
                            start_date=START, end_date=END, adjust="qfq")
    return df
def add_signal(df):
    df['ma5'] = df['收盘'].rolling(5).mean()
    df['ma10'] = df['收盘'].rolling(10).mean()
    df['ma20'] = df['收盘'].rolling(20).mean()
    df['vol_ma5'] = df['成交量'].rolling(5).mean()
    df['vol_ratio'] = df['成交量'] / df['vol_ma5']
    df['昨天的收盘'] = df['收盘'].shift(1)
    df['昨天在20线下'] = df['昨天的收盘'] < df['ma20'].shift(1)
    df['今天在20线上'] = df['收盘'] > df['ma20']
    df['放量'] = df['vol_ratio'] > VOL_RATIO_MIN
    df['信号'] = df['昨天在20线下'] & df['今天在20线上'] & df['放量']
    return df
def backtest(sig_df):
    trade_rows = []
    sig = sig_df[sig_df["信号"] == True].copy()
    for idx, row in sig.iterrows():
        buy_idx = idx + 1
        sell_idx = idx + 5
        if buy_idx >= len(sig_df) or sell_idx >= len(sig_df):
            continue
        buy_price = sig_df.iloc[buy_idx]["收盘"]
        sell_price = sig_df.iloc[sell_idx]["收盘"]
        ret = sell_price / buy_price - 1
        trade_rows.append({
            "信号日": row["日期"],
            "买入日": sig_df.iloc[buy_idx]["日期"],
            "买入价": buy_price,
            "卖出价": sell_price,
            "收益率": ret
        })
    return pd.DataFrame(trade_rows)
trades_list = []
for code, name in STOCKS:
    try:
        raw = fetch_data(code)
        sig = add_signal(raw)
        sig["股票"] = name
        sig["代码"] = code
        res = backtest(sig)
        trades_list.append(res)
        print(f"{code} {name} 完成")
    except Exception as e:
        print(f"{code} {name} 失败: {type(e).__name__}: {e}")
trades = pd.concat(trades_list, ignore_index=True)
print("交易次数：", len(trades))
if len(trades)>0:
    print("平均收益：", round(trades["收益率"].mean(), 4))
    print("胜率：", round(((trades["收益率"] > 0).mean()), 4))
display(trades.head())


600519 贵州茅台 完成
601318 中国平安 完成
300750 宁德时代 完成
002594 比亚迪 完成
交易次数： 63
平均收益： -0.0029
胜率： 0.3968


,信号日,买入日,买入价,卖出价,收益率
0,2023-02-13,2023-02-14,1664.43,1695.64,0.018751
1,2023-02-16,2023-02-17,1641.42,1638.64,-0.001694
2,2023-05-22,2023-05-23,1545.19,1509.64,-0.023007
3,2023-06-12,2023-06-13,1519.64,1564.64,0.029612
4,2023-09-18,2023-09-19,1673.49,1678.93,0.003251
